# Create assumption-aligned external scheduler sequence

This notebook regenerates demo external schedule files for the Blast Master Scenario Simulation workflow. It uses `simple_mine_schedule.xlsx` for scheduling assumptions, `blast_master.csv` for pattern geometry, `block_model_copper.csv` for material context, and `drill_charge_designs.zip` for drill/charge context.


In [ ]:
import csv
import json
import math
import os
import re
import shutil
import zipfile
import io
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

INPUT_DIR = Path(".")  # change this if running from another folder
BLAST_MASTER_CSV = INPUT_DIR / "blast_master.csv"
BLOCK_MODEL_CSV = INPUT_DIR / "block_model_copper.csv"
DRILL_CHARGE_ZIP = INPUT_DIR / "drill_charge_designs.zip"
SCHEDULE_XLSX = INPUT_DIR / "simple_mine_schedule.xlsx"

OUT_DIR = INPUT_DIR / "demo_external_sequence_assumption_aligned"
OUT_DIR.mkdir(exist_ok=True)

def parse_float(v, default=None):
    try:
        if v is None or v == "":
            return default
        return float(v)
    except Exception:
        return default

def safe_div(a, b, default=None):
    try:
        if b and math.isfinite(a) and math.isfinite(b):
            return a / b
    except Exception:
        pass
    return default

def mean(values, default=None):
    vals = [v for v in values if isinstance(v, (int, float)) and math.isfinite(v)]
    return sum(vals) / len(vals) if vals else default

def frac(vals, predicate):
    vals = list(vals)
    return sum(1 for v in vals if predicate(v)) / len(vals) if vals else 0.0

def iso(dt):
    return dt.isoformat().replace("+00:00", "Z")

In [ ]:
# --- Minimal XLSX reader: avoids requiring pandas/openpyxl for this demo notebook.
def _col_to_index(cell_ref):
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    n = 0
    for ch in letters:
        n = n * 26 + (ord(ch.upper()) - 64)
    return n - 1

def read_first_sheet_values(xlsx_path):
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(xlsx_path) as z:
        shared_strings = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", ns):
                texts = [t.text or "" for t in si.findall(".//a:t", ns)]
                shared_strings.append("".join(texts))

        # workbook relationship to first sheet
        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        first_sheet = workbook.find(".//a:sheet", ns)
        rel_id = first_sheet.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id")
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        target = None
        for rel in rels:
            if rel.attrib.get("Id") == rel_id:
                target = rel.attrib["Target"]
                break
        sheet_path = "xl/" + target.lstrip("/")
        root = ET.fromstring(z.read(sheet_path))
        rows = []
        for row in root.findall(".//a:sheetData/a:row", ns):
            row_index = int(row.attrib["r"]) - 1
            while len(rows) <= row_index:
                rows.append([])
            for cell in row.findall("a:c", ns):
                ref = cell.attrib.get("r", "")
                col_index = _col_to_index(ref)
                while len(rows[row_index]) <= col_index:
                    rows[row_index].append(None)
                cell_type = cell.attrib.get("t")
                v = cell.find("a:v", ns)
                if v is None:
                    value = None
                elif cell_type == "s":
                    value = shared_strings[int(v.text)]
                else:
                    raw = v.text
                    try:
                        num = float(raw)
                        value = int(num) if num.is_integer() else num
                    except Exception:
                        value = raw
                rows[row_index][col_index] = value
        # Rectangularize to useful width
        max_cols = max((len(r) for r in rows), default=0)
        for r in rows:
            while len(r) < max_cols:
                r.append(None)
        return rows

sheet = read_first_sheet_values(SCHEDULE_XLSX)

availability_pct = parse_float(sheet[1][2], 85)
utilisation_pct = parse_float(sheet[2][2], 90)
drill_pen_rate_m_hr = parse_float(sheet[3][2], 30)
availability = availability_pct / 100 if availability_pct > 1 else availability_pct
utilisation = utilisation_pct / 100 if utilisation_pct > 1 else utilisation_pct
shift_hours = 12
shifts_per_day = 2
effective_hours_per_day = shift_hours * shifts_per_day * availability * utilisation

schedule_rows = []
for row in sheet[8:]:
    if len(row) < 18 or row[1] is None:
        continue
    try:
        pid_int = int(row[1])
    except Exception:
        continue
    schedule_rows.append({
        "pattern_id": str(pid_int),
        "sequence_index": len(schedule_rows) + 1,
        "area_m2": parse_float(row[2], 0),
        "bench_height_m": parse_float(row[3], 15),
        "volume_m3": parse_float(row[4], 0),
        "dig_rate_bcm_hr": parse_float(row[5], 0),
        "dig_hours": parse_float(row[6], 0),
        "dig_units": parse_float(row[7], 1),
        "excavation_days": parse_float(row[8], 0),
        "cumulative_excavation_days": parse_float(row[9], 0),
        "drill_metres_m": parse_float(row[10], 0),
        "holes": int(round(parse_float(row[11], 0) or 0)),
        "drill_hours": parse_float(row[12], 0),
        "drills": parse_float(row[13], 0),
        "drill_days": parse_float(row[14], 0),
        "cumulative_drill_days": parse_float(row[15], 0),
        "charge_blast_days": parse_float(row[16], 0),
        "cumulative_charge_blast_days": parse_float(row[17], 0),
    })

print(f"Loaded {len(schedule_rows)} schedule rows from {SCHEDULE_XLSX.name}")
print(f"Availability={availability_pct}% Utilisation={utilisation_pct}% Drill penetration={drill_pen_rate_m_hr} m/hr Effective hours/day={effective_hours_per_day:.2f}")

In [ ]:
# Geometry helpers
def polygon_area(points):
    if len(points) < 3:
        return 0.0
    return abs(sum(x1 * y2 - x2 * y1 for (x1, y1), (x2, y2) in zip(points, points[1:] + points[:1]))) / 2.0

def polygon_perimeter(points):
    return sum(math.hypot(x2-x1, y2-y1) for (x1, y1), (x2, y2) in zip(points, points[1:] + points[:1])) if len(points) > 1 else 0.0

def polygon_centroid(points):
    if len(points) < 3:
        return (None, None) if not points else (sum(p[0] for p in points)/len(points), sum(p[1] for p in points)/len(points))
    a = cx = cy = 0.0
    for (x1,y1),(x2,y2) in zip(points, points[1:] + points[:1]):
        cross = x1*y2 - x2*y1
        a += cross
        cx += (x1+x2)*cross
        cy += (y1+y2)*cross
    if abs(a) < 1e-9:
        return (sum(p[0] for p in points)/len(points), sum(p[1] for p in points)/len(points))
    a *= 0.5
    return (cx/(6*a), cy/(6*a))

def point_in_poly(x, y, poly):
    inside = False
    if len(poly) < 3:
        return False
    xj, yj = poly[-1]
    for xi, yi in poly:
        if ((yi > y) != (yj > y)):
            x_inter = (xj-xi)*(y-yi)/(yj-yi+1e-30) + xi
            if x < x_inter:
                inside = not inside
        xj, yj = xi, yi
    return inside

# Load Blast Master polygons
polygons = {}
pattern_types = {}
with open(BLAST_MASTER_CSV, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        pid = str(row["pattern_id"]).strip()
        pattern_types[pid] = row.get("pattern_type", "").strip() or "Production"
        x = parse_float(row.get("x"))
        y = parse_float(row.get("y"))
        if x is not None and y is not None:
            polygons.setdefault(pid, []).append((x, y))

poly_stats = {}
bboxes = {}
for pid, pts in polygons.items():
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    poly_stats[pid] = {
        "area_m2_geom": polygon_area(pts),
        "perimeter_m": polygon_perimeter(pts),
        "centroid_x": polygon_centroid(pts)[0],
        "centroid_y": polygon_centroid(pts)[1],
        "point_count": len(pts),
    }
    bboxes[pid] = (min(xs), min(ys), max(xs), max(ys))

# Load and clip block model to polygons
numeric_fields = ["density_tpm3","cu_pct","au_gpt","ag_gpt","Axb","BWi_kWht","DWi_kWhm3","UCS_MPa","RMR","sulfides_pct","carbonate_pct","clays_pct","fault_dist_m","cn_demand_kgt","rec_au","rec_cu","rec_ag","pf_target_kgpm3","target_p80_mm"]
pattern_blocks = {pid: [] for pid in polygons}
global_blocks = []
with open(BLOCK_MODEL_CSV, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        x = parse_float(row.get("x"))
        y = parse_float(row.get("y"))
        rec = dict(row)
        for k in numeric_fields:
            rec[k] = parse_float(row.get(k))
        global_blocks.append(rec)
        if x is None or y is None:
            continue
        for pid, poly in polygons.items():
            minx, miny, maxx, maxy = bboxes[pid]
            if minx <= x <= maxx and miny <= y <= maxy and point_in_poly(x, y, poly):
                pattern_blocks[pid].append(rec)
                break

global_stats = {
    "density_tpm3": mean([r.get("density_tpm3") for r in global_blocks], 2.7),
    "cu_pct": mean([r.get("cu_pct") for r in global_blocks], 0.0),
    "Axb": mean([r.get("Axb") for r in global_blocks], 50),
    "BWi_kWht": mean([r.get("BWi_kWht") for r in global_blocks], 18),
    "DWi_kWhm3": mean([r.get("DWi_kWhm3") for r in global_blocks], 6),
    "UCS_MPa": mean([r.get("UCS_MPa") for r in global_blocks], 100),
    "rec_cu": mean([r.get("rec_cu") for r in global_blocks], 0.82),
}

block_stats = {}
for pid, rows in pattern_blocks.items():
    stats = {"block_count": len(rows)}
    for k in ["density_tpm3","cu_pct","au_gpt","ag_gpt","Axb","BWi_kWht","DWi_kWhm3","UCS_MPa","RMR","sulfides_pct","carbonate_pct","clays_pct","fault_dist_m","cn_demand_kgt","rec_au","rec_cu","rec_ag","pf_target_kgpm3","target_p80_mm"]:
        stats[k] = mean([r.get(k) for r in rows], global_stats.get(k))
    stats["wet_fraction"] = frac(rows, lambda r: str(r.get("wet","")).lower() in ("true","1","yes","y"))
    for k in ["digability","lithology","mineral_domain","structure_domain","alteration_domain","oxidation","location"]:
        cnt = Counter(str(r.get(k,"")).strip() for r in rows if str(r.get(k,"")).strip())
        stats[k] = cnt.most_common(1)[0][0] if cnt else None
    ore_count = 0
    for r in rows:
        loc = str(r.get("location","")).lower()
        md = str(r.get("mineral_domain","")).lower()
        cu = r.get("cu_pct") or 0
        if ("ore" in loc) or ("ore" in md) or (cu >= 0.15):
            ore_count += 1
    stats["ore_fraction_blocks"] = ore_count / len(rows) if rows else 0.0
    block_stats[pid] = stats

# Load drill/charge design context
design_stats = {}
with zipfile.ZipFile(DRILL_CHARGE_ZIP) as z:
    for name in z.namelist():
        if name.startswith("drill_designs/") and name.endswith(".csv"):
            m = re.search(r"-(\d+)\.csv$", name)
            if not m:
                continue
            pid = str(int(m.group(1)))
            with z.open(name) as f:
                rows = list(csv.DictReader(io.TextIOWrapper(f)))
            if not rows:
                continue
            def avg(field, default=None):
                return mean([parse_float(r.get(field)) for r in rows], default)
            design_stats.setdefault(pid, {}).update({
                "design_hole_count": len(rows),
                "design_drill_metres_m": sum(parse_float(r.get("length_m"), 0) or 0 for r in rows),
                "burden_m": avg("burden_m"),
                "spacing_m": avg("spacing_m"),
                "hole_diameter_mm": avg("hole_diameter_mm"),
                "design_explosive_kg": sum(parse_float(r.get("explosive_mass_kg"), 0) or 0 for r in rows),
                "predicted_p50_mm": avg("predicted_p50_mm"),
                "predicted_p80_mm": avg("predicted_p80_mm"),
                "predicted_p95_mm": avg("predicted_p95_mm"),
                "target_p80_mm": avg("target_p80_mm"),
                "hardness_index": avg("hardness_index"),
                "grade_index": avg("grade_index"),
                "energy_requirement_index": avg("energy_requirement_index"),
            })

    for name in z.namelist():
        if name.startswith("charge_designs/") and name.endswith(".csv"):
            m = re.search(r"-(\d+)\.csv$", name)
            if not m:
                continue
            pid = str(int(m.group(1)))
            with z.open(name) as f:
                rows = list(csv.DictReader(io.TextIOWrapper(f)))
            charge_kg = sum(parse_float(r.get("design_explosive_kg"), 0) or 0 for r in rows)
            products = Counter(r.get("explosive_product","") for r in rows if r.get("explosive_product"))
            design_stats.setdefault(pid, {}).update({
                "charge_design_hole_count": len(rows),
                "charge_design_explosive_kg": charge_kg,
                "explosive_product": products.most_common(1)[0][0] if products else "ANFO",
            })

print(f"Loaded {len(polygons)} polygons, {len(global_blocks)} block-model rows and {len(design_stats)} design summaries.")

In [ ]:
BASE_START = datetime(2026, 6, 1, 6, 0, 0, tzinfo=timezone.utc)

def hardness_factor_from_stats(stats):
    bwi = stats.get("BWi_kWht") or global_stats.get("BWi_kWht") or 18
    ucs = stats.get("UCS_MPa") or global_stats.get("UCS_MPa") or 100
    axb = stats.get("Axb") or global_stats.get("Axb") or 50
    factor = (bwi / 18.0) ** 0.35 * (ucs / 100.0) ** 0.25 * (50.0 / max(axb, 1.0)) ** 0.20
    return max(0.6, min(1.8, factor))

def choose_destination(ptype, grade, ore_frac):
    p = (ptype or "").lower()
    if p in ("ramp", "slot"):
        return "Development / access material"
    if p in ("trim", "drop-cut", "dropcut"):
        return "Boundary control / trim cleanup"
    if grade is None:
        return "ROM stockpile"
    if grade >= 1.0 and ore_frac > 0.5:
        return "Direct crusher feed"
    if grade >= 0.35 and ore_frac > 0.3:
        return "ROM stockpile"
    if grade >= 0.15:
        return "Low-grade stockpile"
    return "Waste dump"

def row_to_pattern(row):
    pid = row["pattern_id"]
    ptype = pattern_types.get(pid, "Production")
    bstats = block_stats.get(pid, {})
    dstats = design_stats.get(pid, {})
    poly = poly_stats.get(pid, {})

    density = bstats.get("density_tpm3") or 2.7
    volume_m3 = row["volume_m3"] or (row["area_m2"] or poly.get("area_m2_geom", 0)) * (row["bench_height_m"] or 15)
    tonnes = volume_m3 * density
    grade = bstats.get("cu_pct")
    ore_frac = bstats.get("ore_fraction_blocks")
    ore_frac = max(0, min(1, ore_frac if ore_frac is not None else (1 if (grade or 0) >= 0.15 else 0)))
    ore_tonnes = tonnes * ore_frac
    waste_tonnes = tonnes - ore_tonnes
    contained_metal = ore_tonnes * (grade or 0) / 100 if grade is not None else None
    recovery_pct = (bstats.get("rec_cu") if bstats.get("rec_cu") is not None else 0.82) * 100
    recovered_metal = contained_metal * recovery_pct / 100 if contained_metal is not None else None

    charge_kg = dstats.get("charge_design_explosive_kg") or dstats.get("design_explosive_kg") or 0.0
    scale_by_m = safe_div(row["drill_metres_m"], dstats.get("design_drill_metres_m") or 0.0, 1.0)
    explosive_kg = charge_kg * (scale_by_m if scale_by_m and scale_by_m > 0 else 1.0)
    pf_kg_m3 = safe_div(explosive_kg, volume_m3, 0.0) or 0.0
    pf_kg_t = safe_div(explosive_kg, tonnes, 0.0) or 0.0

    predicted_p80 = dstats.get("predicted_p80_mm")
    target_p80 = dstats.get("target_p80_mm") or bstats.get("target_p80_mm")
    hardness_factor = hardness_factor_from_stats(bstats)
    digability = bstats.get("digability") or "medium"

    prev_exc_days = row["cumulative_excavation_days"] - row["excavation_days"]
    start = BASE_START + timedelta(days=prev_exc_days)
    finish = BASE_START + timedelta(days=row["cumulative_excavation_days"])
    drill_start = BASE_START + timedelta(days=row["cumulative_drill_days"] - row["drill_days"])
    drill_finish = BASE_START + timedelta(days=row["cumulative_drill_days"])
    charge_start = BASE_START + timedelta(days=row["cumulative_charge_blast_days"] - row["charge_blast_days"])
    charge_finish = BASE_START + timedelta(days=row["cumulative_charge_blast_days"])
    day_index = int(math.floor(prev_exc_days)) + 1
    period_id = f"D{day_index:02d}"

    flags, data_flags, geom_flags, why = [], [], [], []
    if ptype.lower() in ("ramp", "slot", "trim"):
        flags.append("development_or_boundary_control")
        why.append(f"{ptype} pattern preserves access, opening sequence or wall-control readiness.")
    if str(digability).lower() == "hard" or hardness_factor >= 1.15:
        flags.append("hard_digability")
        why.append("Block-model hardness/digability indicates elevated dig and comminution risk.")
    if predicted_p80 and predicted_p80 > 180:
        flags.append("coarse_fragmentation_risk")
        why.append("Predicted P80 is coarse relative to a stable mill-feed target.")
    if bstats.get("wet_fraction", 0) > 0.35:
        flags.append("wet_material_risk")
        why.append("Wet block-model fraction may affect crusher and material-handling stability.")
    if grade and grade >= 1.0:
        flags.append("high_grade_value_priority")
        why.append("High Cu grade makes sequencing and plant-feed stability economically important.")
    if bstats.get("block_count", 0) < 5:
        data_flags.append("limited_block_model_support")
    if row["holes"] != dstats.get("design_hole_count"):
        data_flags.append("schedule_design_hole_count_reconciled")
    if row["area_m2"] < 3000 or ptype.lower() in ("ramp", "slot", "trim"):
        geom_flags.append("small_or_special_pattern")
    if not why:
        why.append("Production pattern scheduled according to the spreadsheet sequence.")

    return {
        "pattern_id": pid,
        "pattern_key": f"A-185-08-{int(pid):02d}",
        "pattern_type": ptype,
        "pit": "Site B / Pit A",
        "mine_area": "Bench RL185 / BM v3",
        "bench": "RL185-200",
        "phase": "Demo Phase A",
        "pushback": "Pit A PB-01",
        "source_system": "Demo external scheduler - simple_mine_schedule.xlsx",
        "schedule_name": "Assumption-aligned demo sequence",
        "planned_loader": "EX-01" if row["dig_units"] <= 1.1 else ("EX-01 + EX-02" if row["dig_units"] <= 2 else "EX fleet"),
        "planned_truck_fleet": "Fleet A" if day_index <= 14 else ("Fleet B" if day_index <= 28 else "Fleet A/B"),
        "destination": choose_destination(ptype, grade, ore_frac),
        "estimated_tonnes": tonnes,
        "ore_tonnes": ore_tonnes,
        "waste_tonnes": waste_tonnes,
        "mineralised_tonnes": ore_tonnes,
        "average_grade": grade,
        "contained_metal_t": contained_metal,
        "recovered_metal_t": recovered_metal,
        "metal": "Cu",
        "value_proxy": recovered_metal,
        "hole_count": row["holes"],
        "design_hole_count": dstats.get("design_hole_count"),
        "drill_metres_m": row["drill_metres_m"],
        "design_drill_metres_m": dstats.get("design_drill_metres_m"),
        "explosive_kg": explosive_kg,
        "charge_design_explosive_kg": charge_kg,
        "explosive_product": dstats.get("explosive_product") or "ANFO",
        "powder_factor_kg_m3": pf_kg_m3,
        "powder_factor_kg_t": pf_kg_t,
        "burden_m": dstats.get("burden_m") or 7.0,
        "spacing_m": dstats.get("spacing_m") or 8.0,
        "hole_diameter_mm": dstats.get("hole_diameter_mm") or 229.0,
        "target_p80_mm": target_p80,
        "predicted_p50_mm": dstats.get("predicted_p50_mm"),
        "predicted_p80_mm": predicted_p80,
        "predicted_p95_mm": dstats.get("predicted_p95_mm"),
        "hardness_index": dstats.get("hardness_index"),
        "hardness_factor": hardness_factor,
        "BWi_kWht": bstats.get("BWi_kWht"),
        "UCS_MPa": bstats.get("UCS_MPa"),
        "Axb": bstats.get("Axb"),
        "DWi_kWhm3": bstats.get("DWi_kWhm3"),
        "digability": digability,
        "primary_mineral_domain": bstats.get("mineral_domain"),
        "primary_lithology": bstats.get("lithology"),
        "structure_domain": bstats.get("structure_domain"),
        "wet_fraction": bstats.get("wet_fraction"),
        "area_m2": row["area_m2"],
        "area_m2_geom": poly.get("area_m2_geom"),
        "perimeter_m": poly.get("perimeter_m"),
        "centroid_x": poly.get("centroid_x"),
        "centroid_y": poly.get("centroid_y"),
        "density_tpm3": density,
        "ore_fraction": ore_frac,
        "volume_m3": volume_m3,
        "bench_height_m": row["bench_height_m"],
        "dig_rate_bcm_hr": row["dig_rate_bcm_hr"],
        "predicted_dig_rate_tph": row["dig_rate_bcm_hr"] * density if row["dig_rate_bcm_hr"] else None,
        "dig_hours": row["dig_hours"],
        "dig_units": row["dig_units"],
        "excavation_days": row["excavation_days"],
        "cumulative_excavation_days": row["cumulative_excavation_days"],
        "drill_hours": row["drill_hours"],
        "drills": row["drills"],
        "drill_days": row["drill_days"],
        "cumulative_drill_days": row["cumulative_drill_days"],
        "charge_blast_days": row["charge_blast_days"],
        "cumulative_charge_blast_days": row["cumulative_charge_blast_days"],
        "haul_distance_m": 1100 + day_index * 22 + (80 if ptype.lower() in ("trim", "ramp", "slot") else 0),
        "truck_hours": row["dig_hours"] * (1.5 + 0.15 * min(day_index, 30) / 30),
        "loader_hours": row["dig_hours"],
        "drill_duration_hr": row["drill_hours"],
        "charge_duration_hr": row["charge_blast_days"] * 24 * availability * utilisation,
        "load_haul_duration_hr": row["dig_hours"],
        "total_duration_hr": row["drill_hours"] + row["charge_blast_days"] * 24 * availability * utilisation + row["dig_hours"],
        "development_priority": 100 if ptype.lower() in ("ramp", "slot") else (70 if ptype.lower() == "trim" else 30),
        "access_priority": 1 if ptype.lower() in ("ramp", "slot") else (2 if row["sequence_index"] <= 12 else 3),
        "dependency_pattern_ids": [] if row["sequence_index"] == 1 else [str(row["sequence_index"] - 1)],
        "unlocks_pattern_ids": [str(row["sequence_index"] + 1)] if row["sequence_index"] < len(schedule_rows) else [],
        "is_isolated": False,
        "risk_flags": flags,
        "data_quality_flags": data_flags,
        "geometry_flags": geom_flags,
        "why": why,
        "sequence_index": row["sequence_index"],
        "period_id": period_id,
        "period_label": f"Day {day_index}",
        "scheduled_period": period_id,
        "scheduled_start": iso(start),
        "scheduled_finish": iso(finish),
        "drill_start": iso(drill_start),
        "drill_finish": iso(drill_finish),
        "charge_blast_start": iso(charge_start),
        "charge_blast_finish": iso(charge_finish),
        "schedule_assumption_availability_pct": availability_pct,
        "schedule_assumption_utilisation_pct": utilisation_pct,
        "schedule_assumption_drill_penetration_rate_m_hr": drill_pen_rate_m_hr,
        "effective_hours_per_day": effective_hours_per_day,
    }

all_records = [row_to_pattern(r) for r in schedule_rows]
records_2week = [r for r in all_records if r["cumulative_excavation_days"] <= 15.0]
records_4week = [r for r in all_records if r["cumulative_excavation_days"] <= 29.0]
print(f"Prepared: 2-week={len(records_2week)}, 4-week={len(records_4week)}, full-bench={len(all_records)}")

In [ ]:
FIELD_ORDER = [
    "sequence_index","period_id","period_label","scheduled_period","scheduled_start","scheduled_finish",
    "drill_start","drill_finish","charge_blast_start","charge_blast_finish",
    "pattern_id","pattern_key","pattern_type","pit","mine_area","bench","phase","pushback","source_system","schedule_name",
    "planned_loader","planned_truck_fleet","destination",
    "estimated_tonnes","ore_tonnes","waste_tonnes","mineralised_tonnes","average_grade","contained_metal_t","recovered_metal_t","metal","value_proxy",
    "volume_m3","area_m2","area_m2_geom","bench_height_m","density_tpm3","ore_fraction",
    "hole_count","design_hole_count","drill_metres_m","design_drill_metres_m","drill_hours","drills","drill_days","cumulative_drill_days",
    "explosive_kg","charge_design_explosive_kg","explosive_product","charge_blast_days","cumulative_charge_blast_days",
    "powder_factor_kg_m3","powder_factor_kg_t","burden_m","spacing_m","hole_diameter_mm",
    "target_p80_mm","predicted_p50_mm","predicted_p80_mm","predicted_p95_mm",
    "hardness_index","hardness_factor","BWi_kWht","UCS_MPa","Axb","DWi_kWhm3","digability",
    "primary_mineral_domain","primary_lithology","structure_domain","wet_fraction",
    "dig_rate_bcm_hr","predicted_dig_rate_tph","dig_hours","dig_units","excavation_days","cumulative_excavation_days",
    "haul_distance_m","truck_hours","loader_hours","drill_duration_hr","charge_duration_hr","load_haul_duration_hr","total_duration_hr",
    "development_priority","access_priority","dependency_pattern_ids","unlocks_pattern_ids","is_isolated",
    "risk_flags","data_quality_flags","geometry_flags","why",
    "perimeter_m","centroid_x","centroid_y",
    "schedule_assumption_availability_pct","schedule_assumption_utilisation_pct","schedule_assumption_drill_penetration_rate_m_hr","effective_hours_per_day"
]

def csv_value(v):
    if isinstance(v, list):
        return " | ".join(str(x) for x in v)
    if isinstance(v, dict):
        return json.dumps(v, separators=(",", ":"))
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    return v

def weighted_avg(records, value_key, weight_key):
    total_w = total_v = 0.0
    for r in records:
        v = r.get(value_key)
        w = r.get(weight_key) or 0
        if v is not None and isinstance(v, (int, float)) and math.isfinite(v) and w > 0:
            total_v += v * w
            total_w += w
    return total_v / total_w if total_w else None

def aggregate_summary(records, name):
    total_tonnes = sum((r.get("estimated_tonnes") or 0) for r in records)
    ore_tonnes = sum((r.get("ore_tonnes") or 0) for r in records)
    contained = sum((r.get("contained_metal_t") or 0) for r in records)
    risk = Counter(flag for r in records for flag in r.get("risk_flags", []))
    return {
        "name": name,
        "pattern_count": len(records),
        "total_tonnes": total_tonnes,
        "ore_tonnes": ore_tonnes,
        "waste_tonnes": sum((r.get("waste_tonnes") or 0) for r in records),
        "average_grade": contained * 100 / ore_tonnes if ore_tonnes else None,
        "contained_metal_t": contained,
        "recovered_metal_t": sum((r.get("recovered_metal_t") or 0) for r in records),
        "total_volume_m3": sum((r.get("volume_m3") or 0) for r in records),
        "total_drill_metres_m": sum((r.get("drill_metres_m") or 0) for r in records),
        "total_explosive_kg": sum((r.get("explosive_kg") or 0) for r in records),
        "weighted_density_tpm3": weighted_avg(records, "density_tpm3", "estimated_tonnes"),
        "weighted_predicted_p80_mm": weighted_avg(records, "predicted_p80_mm", "estimated_tonnes"),
        "weighted_dig_rate_bcm_hr": weighted_avg(records, "dig_rate_bcm_hr", "estimated_tonnes"),
        "weighted_predicted_dig_rate_tph": weighted_avg(records, "predicted_dig_rate_tph", "estimated_tonnes"),
        "weighted_bwi_kwht": weighted_avg(records, "BWi_kWht", "estimated_tonnes"),
        "weighted_ucs_mpa": weighted_avg(records, "UCS_MPa", "estimated_tonnes"),
        "calendar_excavation_days": max((r.get("cumulative_excavation_days") or 0) for r in records) if records else 0,
        "calendar_drill_days": max((r.get("cumulative_drill_days") or 0) for r in records) if records else 0,
        "calendar_charge_blast_days": max((r.get("cumulative_charge_blast_days") or 0) for r in records) if records else 0,
        "effective_hours_per_day": effective_hours_per_day,
        "availability_pct": availability_pct,
        "utilisation_pct": utilisation_pct,
        "drill_penetration_rate_m_hr": drill_pen_rate_m_hr,
        "pattern_type_counts": dict(Counter(r.get("pattern_type") for r in records)),
        "risk_flag_counts": dict(risk),
    }

def period_summaries(records):
    groups = defaultdict(list)
    for r in records:
        groups[r["scheduled_period"]].append(r)
    summaries = []
    for pid in sorted(groups, key=lambda x: int(x[1:]) if x[1:].isdigit() else x):
        rows = groups[pid]
        starts = [datetime.fromisoformat(r["scheduled_start"].replace("Z", "+00:00")) for r in rows]
        finishes = [datetime.fromisoformat(r["scheduled_finish"].replace("Z", "+00:00")) for r in rows]
        s = aggregate_summary(rows, pid)
        s.update({
            "period_id": pid,
            "period_label": f"Day {int(pid[1:])}" if pid[1:].isdigit() else pid,
            "period_start": iso(min(starts)),
            "period_finish": iso(max(finishes)),
            "pattern_ids": [r["pattern_id"] for r in rows],
            "destination_counts": dict(Counter(r.get("destination") for r in rows)),
            "primary_destination": Counter(r.get("destination") for r in rows).most_common(1)[0][0] if rows else None,
        })
        summaries.append(s)
    return summaries

def make_asset(records, schedule_id, horizon, name, filename):
    generated_at = iso(datetime.now(timezone.utc))
    summary = aggregate_summary(records, name)
    ps = period_summaries(records)
    return {
        "stage": "blast_master_schedule",
        "schedule_id": schedule_id,
        "source_system": "Demo external scheduler - simple_mine_schedule.xlsx",
        "source_file": "simple_mine_schedule.xlsx",
        "generated_at": generated_at,
        "horizon": horizon,
        "period_granularity": "day",
        "objective": "meet_targets",
        "constraints": {
            "start_date_iso": iso(BASE_START),
            "horizon": horizon,
            "granularity": "day",
            "source_file": "simple_mine_schedule.xlsx",
            "availability_pct": availability_pct,
            "utilisation_pct": utilisation_pct,
            "drill_penetration_rate_m_hr": drill_pen_rate_m_hr,
            "shift_hours": shift_hours,
            "shifts_per_day": shifts_per_day,
            "effective_hours_per_day": effective_hours_per_day,
            "basis": "Spreadsheet cumulative drill, charge/blast and excavation days drive sequencing. Block model and drill/charge designs provide downstream context.",
        },
        "patterns": records,
        "period_summaries": ps,
        "summary": summary,
        "scenario": {
            "scenarioId": schedule_id.replace(":", "-"),
            "name": name,
            "generatedAt": generated_at,
            "objective": "meet_targets",
            "horizon": horizon,
            "granularity": "day",
            "sourceFile": filename,
            "summary": summary,
            "periods": [
                {
                    "periodId": p["period_id"],
                    "label": p["period_label"],
                    "start": p["period_start"],
                    "finish": p["period_finish"],
                    "rows": [r for r in records if r["scheduled_period"] == p["period_id"]],
                }
                for p in ps
            ],
        },
        "external_schedule_metadata": {
            "intended_use": "Demo sequence input for Blast Master Scenario Simulation.",
            "not_a_vendor_export": True,
            "assumption_workbook": "simple_mine_schedule.xlsx",
            "input_files": ["blast_master.csv", "block_model_copper.csv", "drill_charge_designs.zip", "simple_mine_schedule.xlsx"],
            "assumption_notes": [
                "Availability, utilisation and drill penetration rate are read from the workbook.",
                "Pattern order, excavation days, drill days and charge/blast days are read from the workbook.",
                "Tonnes, grade, hardness and recovery are derived from the block model clipped to Blast Master polygons.",
                "Explosive mass is scaled from charge design files to the workbook drill-metres basis.",
            ],
        },
    }

def write_csv(records, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELD_ORDER, extrasaction="ignore")
        writer.writeheader()
        for r in records:
            writer.writerow({k: csv_value(r.get(k)) for k in FIELD_ORDER})

def write_periods(summaries, path):
    cols = ["period_id","period_label","period_start","period_finish","pattern_count","pattern_ids","total_tonnes","ore_tonnes","waste_tonnes","average_grade","contained_metal_t","recovered_metal_t","total_volume_m3","total_drill_metres_m","total_explosive_kg","weighted_density_tpm3","weighted_predicted_p80_mm","weighted_dig_rate_bcm_hr","weighted_predicted_dig_rate_tph","weighted_bwi_kwht","weighted_ucs_mpa","primary_destination","destination_counts","risk_flag_counts","calendar_excavation_days","calendar_drill_days","calendar_charge_blast_days"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        writer.writeheader()
        for r in summaries:
            writer.writerow({k: csv_value(r.get(k)) for k in cols})

def write_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

sets = {
    "2week": (records_2week, "2_weeks", "Assumption-aligned 2-week demo sequence"),
    "4week": (records_4week, "1_month", "Assumption-aligned 4-week demo sequence"),
    "full_bench": (all_records, "demo_52_days", "Assumption-aligned full-bench sequence"),
}

assets = {}
for key, (records, horizon, name) in sets.items():
    asset = make_asset(records, f"external:simple_mine_schedule:{key}:2026-06-01", horizon, name, f"demo_external_scheduler_sequence_{key}.csv")
    assets[key] = asset
    write_csv(records, OUT_DIR / f"demo_external_scheduler_sequence_{key}.csv")
    write_periods(asset["period_summaries"], OUT_DIR / f"demo_external_scheduler_periods_{key}.csv")
    write_json(asset, OUT_DIR / f"demo_external_scheduler_schedule_asset_{key}.json")

# Backward-compatible primary files use the 4-week schedule.
shutil.copyfile(OUT_DIR / "demo_external_scheduler_sequence_4week.csv", OUT_DIR / "demo_external_scheduler_sequence.csv")
shutil.copyfile(OUT_DIR / "demo_external_scheduler_periods_4week.csv", OUT_DIR / "demo_external_scheduler_periods.csv")
shutil.copyfile(OUT_DIR / "demo_external_scheduler_schedule_asset_4week.json", OUT_DIR / "demo_external_scheduler_schedule_asset.json")

# Raw assumptions table
assumption_cols = ["pattern_id","sequence_index","area_m2","bench_height_m","volume_m3","dig_rate_bcm_hr","dig_hours","dig_units","excavation_days","cumulative_excavation_days","drill_metres_m","holes","drill_hours","drills","drill_days","cumulative_drill_days","charge_blast_days","cumulative_charge_blast_days"]
with open(OUT_DIR / "schedule_assumptions_from_simple_mine_schedule.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=assumption_cols)
    writer.writeheader()
    for r in schedule_rows:
        writer.writerow({k: r.get(k) for k in assumption_cols})

write_json({
    "source": "simple_mine_schedule.xlsx",
    "availability_pct": availability_pct,
    "utilisation_pct": utilisation_pct,
    "drill_penetration_rate_m_hr": drill_pen_rate_m_hr,
    "shift_hours": shift_hours,
    "shifts_per_day": shifts_per_day,
    "effective_hours_per_day": effective_hours_per_day,
    "parsed_pattern_count": len(schedule_rows),
    "four_week_cutoff": {"pattern_count": len(records_4week), "last_pattern_id": records_4week[-1]["pattern_id"], "calendar_excavation_days": records_4week[-1]["cumulative_excavation_days"]},
    "two_week_cutoff": {"pattern_count": len(records_2week), "last_pattern_id": records_2week[-1]["pattern_id"], "calendar_excavation_days": records_2week[-1]["cumulative_excavation_days"]},
    "full_bench": {"pattern_count": len(all_records), "calendar_excavation_days": all_records[-1]["cumulative_excavation_days"]},
}, OUT_DIR / "schedule_assumptions_summary.json")

for key, asset in assets.items():
    s = asset["summary"]
    print(f"{key}: {s['pattern_count']} patterns, {s['calendar_excavation_days']:.1f} days, {s['total_tonnes']:,.0f} t, avg Cu {s['average_grade']:.2f}%")
print(f"Wrote files to {OUT_DIR.resolve()}")